# Библиотеки

In [5]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
import plotly.graph_objs as go

import requests
import apimoex
import datetime

import vectorbt as vbt
import ipywidgets as widgets

# Выгрузка данных

In [6]:
# Период времени
start= datetime.datetime(2024,1,1)
finish = datetime.datetime(2024, 12,31)
# Чальный капитал
money = 1_000_000
# Если цена акции упадет на 5% ниже цены покупки, позиция будет автоматически закрыта
stop_loss = 0.05

fast_window = 30
slow_window = 90

In [7]:
# Выгрузка часовых данных о привилегированных акциях Сургутнефтегаза
with requests.Session() as session:
    data = apimoex.get_board_candles(session, 'SNGSP', interval=60, start=start, end=finish)
    df = pd.DataFrame(data)
    df.set_index(pd.to_datetime(df['begin']), inplace=True)
    df.drop(['begin', 'value', 'volume'], axis=1,inplace=True)
df

,open,close,high,low
begin,,,,
2024-01-03 09:00:00,55.350,55.350,55.350,55.350
2024-01-03 10:00:00,55.360,55.650,55.650,55.305
2024-01-03 11:00:00,55.655,55.730,55.900,55.580
2024-01-03 12:00:00,55.760,55.655,55.765,55.485
2024-01-03 13:00:00,55.635,55.650,55.690,55.540
...,...,...,...,...
2024-12-30 19:00:00,60.445,60.255,60.475,60.175
2024-12-30 20:00:00,60.260,60.310,60.335,60.220
2024-12-30 21:00:00,60.320,60.440,60.445,60.280


# Торговля по сигналам

In [8]:
# Скользящие среднии
fast_ma = vbt.MA.run(df['open'], fast_window)
slow_ma = vbt.MA.run(df['open'], slow_window)

# Сигналы при пересечении скользящих средних
ma_entries = fast_ma.ma_crossed_above(slow_ma)
ma_exits = fast_ma.ma_crossed_below(slow_ma)

In [9]:
portfolio = vbt.Portfolio.from_signals(df['close'], ma_entries, ma_exits,
                                       low=df['low'],open=df['open'],
                                       sl_stop=stop_loss, init_cash=money)
# portfolio.logs.records
pf = portfolio.orders.records_readable
pf

,Order Id,Column,Timestamp,Size,Price,Fees,Side
0,0,0,2024-01-17 10:00:00,17652.250662,56.65000,0.0,Buy
1,1,0,2024-01-23 20:00:00,17652.250662,56.13500,0.0,Sell
2,2,0,2024-01-25 17:00:00,17411.862430,56.91000,0.0,Buy
3,3,0,2024-01-31 15:00:00,17411.862430,56.13000,0.0,Sell
4,4,0,2024-02-01 19:00:00,17075.702598,57.23500,0.0,Buy
5,5,0,2024-02-12 12:00:00,17075.702598,57.08500,0.0,Sell
6,6,0,2024-02-13 17:00:00,16718.402930,58.30500,0.0,Buy
7,7,0,2024-02-21 23:00:00,16718.402930,59.70000,0.0,Sell
8,8,0,2024-02-26 15:00:00,16194.850802,61.63000,0.0,Buy
9,9,0,2024-03-06 12:00:00,16194.850802,61.54000,0.0,Sell


In [10]:
stop_exits = ma_entries.vbt.signals.generate_ohlc_stop_exits(
    df.get('open'),
    df.get('high'),
    df.get('low'),
    df.get('close'),
    sl_stop=stop_loss)

only_stop_exits = stop_exits.copy()
only_ma_exits = ma_exits.copy()
prev = 'close'

for (index, entries), ma, stop in zip(ma_entries.items(), ma_exits, stop_exits):
    if stop and prev == 'open':
        only_stop_exits[index] = True
        only_ma_exits[index] = False
        prev = 'close'
    elif ma and prev == 'open':
        only_ma_exits[index] = True
        prev = 'close'
    elif entries and prev == 'close':
        prev = 'open'
        only_stop_exits[index] = False
        only_ma_exits[index] = False
    else:
        only_stop_exits[index] = False
        only_ma_exits[index] = False

In [11]:
trace = go.Candlestick(x=df.index, open=df['open'], high=df['high'],
                low=df['low'], close=df['close'])

fig = go.Figure(trace)
fast_ma.ma.vbt.plot(trace_kwargs=dict(name='Fast MA'), fig=fig)
slow_ma.ma.vbt.plot(trace_kwargs=dict(name='Slow MA'), fig=fig)

ma_entries.vbt.signals.plot_as_entry_markers(df['close'], fig=fig, trace_kwargs=dict(name='sig_buy'))
only_ma_exits.vbt.signals.plot_as_exit_markers(df['close'], fig=fig, trace_kwargs=dict(name='sig_sale'))
only_stop_exits.vbt.signals.plot_as_markers(df['low'], fig=fig,
                                            trace_kwargs=dict(marker=dict(color='black', symbol="triangle-down"),
                                                             name='stop-loss'))
fig.update_layout(width=750, height=700)
iplot(fig)

In [12]:
portfolio.plot_net_exposure().show()

## Отсчётик №1

In [13]:
print('Всего сделок:', pf['Side'].count())
print('Сделки на покупку:', pf[pf['Side']=='Buy']['Side'].count())
print('Сделки на продажу:', pf[pf['Side']=='Sell']['Side'].count())
print('Прибыль:', portfolio.total_profit())

Всего сделок: 45
Сделки на покупку: 23
Сделки на продажу: 22
Прибыль: 191798.72115265753


# Оптимизация

In [14]:
min_window = 2
max_window = 200

In [15]:
comb_fast_ma, comb_slow_ma = vbt.MA.run_combs(df['open'], np.arange(min_window, max_window+1),
                                    r=2, short_names=['fast_ma', 'slow_ma'])

comb_ma_entries = comb_fast_ma.ma_crossed_above(comb_slow_ma)
comb_ma_exits = comb_fast_ma.ma_crossed_below(comb_slow_ma)

In [16]:
comb_portfolio = vbt.Portfolio.from_signals(df['close'], comb_ma_entries, comb_ma_exits,
                                            low=df['low'], open=df['open'],
                                            sl_stop=stop_loss, init_cash=money)
profit = comb_portfolio.total_profit()

In [17]:
profit_matrix = profit.vbt.unstack_to_df(symmetric=True, index_levels='fast_ma_window',
                                         column_levels='slow_ma_window')

fig = go.Figure(data=go.Heatmap(z=profit_matrix.values,
                                x=profit_matrix.columns,
                                y=profit_matrix.index, colorscale = 'haline'))

fig.update_layout(width=700, height=700,
                  xaxis_title='Fast', yaxis_title='Slow')
fig.show()

In [18]:
print('Оптимальные окна:', profit.idxmax())
print('Максимальная прибыль:', profit.max())

Оптимальные окна: (np.int64(111), np.int64(116))
Максимальная прибыль: 513791.799773443


In [19]:
fast_window_opt = profit.idxmax()[0]
slow_window_opt = profit.idxmax()[1]

fast_ma_opt = vbt.MA.run(df['open'], fast_window_opt)
slow_ma_opt = vbt.MA.run(df['open'], slow_window_opt)

ma_entries_opt = fast_ma_opt.ma_crossed_above(slow_ma_opt)
ma_exits_opt = fast_ma_opt.ma_crossed_below(slow_ma_opt)

In [20]:
portfolio_opt = vbt.Portfolio.from_signals(df['close'], ma_entries_opt, ma_exits_opt,
                                           low=df['low'], open=df['open'],
                                           sl_stop=stop_loss, init_cash=money)
pf_opt = portfolio_opt.orders.records_readable

In [21]:
stop_exits_opt = ma_entries_opt.vbt.signals.generate_ohlc_stop_exits(
    df.get('open'),
    df.get('high'),
    df.get('low'),
    df.get('close'),
    sl_stop=stop_loss)

only_stop_exits_opt = stop_exits_opt.copy()
only_ma_exits_opt = ma_exits_opt.copy()
prev = 'close'

for (index, entries), ma, stop in zip(ma_entries_opt.items(), ma_exits_opt, stop_exits_opt):
    if stop and prev == 'open':
        only_stop_exits_opt[index] = True
        only_ma_exits_opt[index] = False
        prev = 'close'
    elif ma and prev == 'open':
        only_ma_exits_opt[index] = True
        prev = 'close'
    elif entries and prev == 'close':
        prev = 'open'
        only_stop_exits_opt[index] = False
        only_ma_exits_opt[index] = False
    else:
        only_stop_exits_opt[index] = False
        only_ma_exits_opt[index] = False

In [22]:
trace = go.Candlestick(x=df.index, open=df['open'], high=df['high'],
                low=df['low'], close=df['close'])

fig = go.Figure(trace)
fast_ma_opt.ma.vbt.plot(trace_kwargs=dict(name='Fast MA'), fig=fig)
slow_ma_opt.ma.vbt.plot(trace_kwargs=dict(name='Slow MA'), fig=fig)

ma_entries_opt.vbt.signals.plot_as_entry_markers(df['close'], fig=fig, trace_kwargs=dict(name='sig_buy'))
only_ma_exits_opt.vbt.signals.plot_as_exit_markers(df['close'], fig=fig, trace_kwargs=dict(name='sig_sale'))
only_stop_exits_opt.vbt.signals.plot_as_markers(df['low'], fig=fig,
                                            trace_kwargs=dict(marker=dict(color='black', symbol="triangle-down"),
                                                             name='stop-loss'))
fig.update_layout(width=750, height=700)
iplot(fig)

## Отсчётик №2

In [23]:
print('Оптимальные окна:', profit.idxmax())
print('Всего сделок (опт):', pf_opt['Side'].count())
print('Сделки на покупку (опт):', pf_opt[pf_opt['Side']=='Buy']['Side'].count())
print('Сделки на продажу (опт):', pf_opt[pf_opt['Side']=='Sell']['Side'].count())
print('Максимальная прибыль:', portfolio_opt.total_profit())

Оптимальные окна: (np.int64(111), np.int64(116))
Всего сделок (опт): 89
Сделки на покупку (опт): 45
Сделки на продажу (опт): 44
Максимальная прибыль: 513791.799773443
